In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_config"

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_1_develop_sa/NBA 3.1 - Dev/02_Input_Data_Creation/00_event_list_config"

In [0]:
import datetime as _dtm

# Run start (UTC, ms) -- used at the end to prove nothing in the 3.0 Volume was modified by this run
_RUN_START_MS = int(_dtm.datetime.now(_dtm.timezone.utc).timestamp() * 1000)
print("Run start (UTC):", _dtm.datetime.fromtimestamp(_RUN_START_MS / 1000, _dtm.timezone.utc))

# Snapshot every simple global set by the 3.0 00_config / 00_event_list_config, so we can detect
# what the %run of 00_cohort_generation (and the base 00_cohort_config it loads) overwrites.
_SIMPLE = (str, int, float, bool, _dtm.date, list, tuple, dict, type(None))
_PRE_COHORT_VARS = {k: repr(v) for k, v in globals().items()
                    if not k.startswith("_") and k not in ("In", "Out") and isinstance(v, _SIMPLE)}
_WATCH = ["OFFSET_DAY", "OFFSET_DAYS", "COHORT_ORIGIN", "COHORT_LAST", "DATA_START_DATE", "dates",
          "lookback_start", "lookback_end", "prediction_start", "prediction_end", "hcp_universe_path",
          "VOLUME_31", "QUERY_OUTPUT_31", "INPUT_PATHS_31", "EVENT_LIST_FILES"]
_PRE_FUNCS = {k: id(globals().get(k)) for k in ["min", "max", "sum", "F", "Get_Data_Snowflakes", "load_event_list", "apply_event_list"]}
print("Snapshot taken:", len(_PRE_COHORT_VARS), "simple globals")

In [0]:
_post = {k: repr(v) for k, v in globals().items()
         if not k.startswith("_") and k not in ("In", "Out") and isinstance(v, _SIMPLE)}
_changed = sorted(k for k in _PRE_COHORT_VARS if k in _post and _post[k] != _PRE_COHORT_VARS[k])
_added = sorted(k for k in _post if k not in _PRE_COHORT_VARS)
_funcs_changed = sorted(k for k, i in _PRE_FUNCS.items() if id(globals().get(k)) != i)

print("=== Variables CHANGED by %run 00_cohort_generation:", _changed or "none")
for k in _changed:
    print(f"  {k}: {_PRE_COHORT_VARS[k][:120]}  ->  {_post[k][:120]}")
print("=== Watched variables (before -> after):")
for k in _WATCH:
    b, a = _PRE_COHORT_VARS.get(k, "<unset>"), _post.get(k, "<unset>")
    print(f"  {k}: {b[:80]} -> {a[:80]}" + ("   <-- CHANGED" if b != a else ""))
print("=== Functions/modules rebound:", _funcs_changed or "none")
print("=== New simple globals added:", len(_added), _added[:40])

# Hard stop if anything the input pull depends on was overwritten
for k in ["VOLUME_31", "QUERY_OUTPUT_31", "INPUT_PATHS_31", "EVENT_LIST_FILES"]:
    assert _PRE_COHORT_VARS.get(k) == _post.get(k), f"{k} was overwritten by the cohort config load"
assert not set(_funcs_changed) & {"Get_Data_Snowflakes", "load_event_list", "apply_event_list", "F"}, \
    f"cohort config load rebound: {_funcs_changed}"

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_1_develop_sa/NBA 3.1 - Dev/00_Helper_Notebooks/00_cohort_generation"

In [0]:
_post = {k: repr(v) for k, v in globals().items()
         if not k.startswith("_") and k not in ("In", "Out") and isinstance(v, _SIMPLE)}
_changed = sorted(k for k in _PRE_COHORT_VARS if k in _post and _post[k] != _PRE_COHORT_VARS[k])
_added = sorted(k for k in _post if k not in _PRE_COHORT_VARS)
_funcs_changed = sorted(k for k, i in _PRE_FUNCS.items() if id(globals().get(k)) != i)

print("=== Variables CHANGED by %run 00_cohort_generation:", _changed or "none")
for k in _changed:
    print(f"  {k}: {_PRE_COHORT_VARS[k][:120]}  ->  {_post[k][:120]}")
print("=== Watched variables (before -> after):")
for k in _WATCH:
    b, a = _PRE_COHORT_VARS.get(k, "<unset>"), _post.get(k, "<unset>")
    print(f"  {k}: {b[:80]} -> {a[:80]}" + ("   <-- CHANGED" if b != a else ""))
print("=== Functions/modules rebound:", _funcs_changed or "none")
print("=== New simple globals added:", len(_added), _added[:40])

# Hard stop if anything the input pull depends on was overwritten
for k in ["VOLUME_31", "QUERY_OUTPUT_31", "INPUT_PATHS_31", "EVENT_LIST_FILES"]:
    assert _PRE_COHORT_VARS.get(k) == _post.get(k), f"{k} was overwritten by the cohort config load"
assert not set(_funcs_changed) & {"Get_Data_Snowflakes", "load_event_list", "apply_event_list", "F"}, \
    f"cohort config load rebound: {_funcs_changed}"

In [0]:
# NBA 3.1: the pull window comes from the claims cohort schedule (c1..c28), NOT from
# get_latest_approved_dates() in the 3.0 00_config (that is the 3.0 one-year inference window).
from dateutil.relativedelta import relativedelta

_claims_cohorts = CohortGenerator("claims").load()
assert len(_claims_cohorts) == 28, f"expected 28 claims cohorts, got {len(_claims_cohorts)}"

# Windowed files start ONE MONTH before the earliest cohort lookback (c1: 2023-06-01 -> 2023-05-01),
# because the 12M AVG features use N+1 = 13 months (AVG_EXTRA_MONTH = True).
# Full-history files have no start date and are not affected.
lookback_start = _claims_cohorts["lookback_start_date"].min() - relativedelta(months=1)
# lookback_end_date is the 1st of the LAST included month (month-boundary semantics), so the
# last included day is the end of that month (c28: 2026-08-01 -> 2026-08-31).
lookback_end = _claims_cohorts["lookback_end_date"].max() + relativedelta(months=1, days=-1)

print(f"NBA 3.1 claims window: {lookback_start} -> {lookback_end} "
      f"({_claims_cohorts.cohort_id.iloc[0]}..{_claims_cohorts.cohort_id.iloc[-1]})")

In [0]:
# Only the cells needed to build the nine patient-feature input files do real work.
# Optional diagnostics (shape/distinct-count prints, displays, DX provider-null breakdown, lab date range)
# each re-pull the data from Snowflake (no caching), so they are off by default.
# The event-list guards (apply_event_list unmatched-share check, validate_event_input) always run,
# and the final verification cell reports row counts, date ranges and universe overlap per file.
RUN_DIAGNOSTICS = False

In [0]:
# Skip any input file whose latest Delta commit was written by this job run (resume after a failure).
# Set to None to pull and rewrite all nine files.
SKIP_EXISTING_FROM_RUN = "5189509327097"
_SKIP_CURRENT = False   # set per section by the "Skip check" cells

def _skip(key):
    """True if INPUT_PATHS_31[key] was last written (WRITE) by job run SKIP_EXISTING_FROM_RUN."""
    if not SKIP_EXISTING_FROM_RUN:
        return False
    try:
        h = spark.sql(f"DESCRIBE HISTORY delta.`{INPUT_PATHS_31[key]}` LIMIT 1").first()
    except Exception:
        h = None
    ok = (h is not None and h.operation == "WRITE" and h.job is not None
          and str(h.job.jobRunId) == str(SKIP_EXISTING_FROM_RUN))
    print(f"[{key}] " + (f"SKIP - already written by run {SKIP_EXISTING_FROM_RUN} at {h.timestamp}" if ok else "PULL"))
    return ok

# While a section is skipped, the event-list helpers do no work (no Snowflake action is triggered);
# the section's write cell is also skipped, so nothing is rewritten.
_apply_event_list_full = globals().get("_apply_event_list_full", apply_event_list)
_validate_event_input_full = globals().get("_validate_event_input_full", validate_event_input)

def apply_event_list(claims, *args, **kwargs):
    return claims if _SKIP_CURRENT else _apply_event_list_full(claims, *args, **kwargs)

def validate_event_input(df, *args, **kwargs):
    return None if _SKIP_CURRENT else _validate_event_input_full(df, *args, **kwargs)

In [0]:
lookback_start, lookback_end

## HCP Universe 

In [0]:
# NBA 3.1 HCP universe (folder, parquet). Replaces the 3.0 hcp_universe_path from the 3.0 00_config
# (kerendia_nba_3_hcp_model_base_hcp_universe_for_inference).
HCP_UNIVERSE_PATH_31 = f"{VOLUME_31}/HCP_Universe"
hcp_universe = (spark.read.parquet(HCP_UNIVERSE_PATH_31)
                .select(F.col("PROVIDER_ID").cast("string").alias("PROVIDER_ID"))
                .distinct())
print("HCP Universe (3.1):", hcp_universe.count())


def in_universe(df):
    """Keep only claims of 3.1-universe HCPs. PROVIDER_ID is a string on both sides."""
    return (df.withColumn("PROVIDER_ID", F.col("PROVIDER_ID").cast("string"))
              .join(F.broadcast(hcp_universe), on="PROVIDER_ID", how="inner"))

### 1. Rx Input Claims

<I>(PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FCT_RX) </I>

##### File 1: Input Data - For 12 months Based Features

In [0]:
_SKIP_CURRENT = _skip("rx_12m")

In [0]:
# NBA 3.1 — Rx 12M: pull all PD claims with NDC_CD in the date window, then join the NDC crosswalk and apply the 3.1 event list (filter + club)
rx_xw = rx_ndc_crosswalk()  # NDC_CD -> PRODUCT_NAME for the 3.1 product list

rx_query = f"""
            SELECT DISTINCT A.CLAIM_ID, A.PROVIDER_ID, A.PATIENT_ID, A.SVC_DT, A.DAYS_SUPPLY, A.NDC_CD, A.REFILL_CODE, A.REJECT_CODE, A.PAYER_PLAN_ID, A.SOB, CLAIM_TYPE, DAW_CODE, ZIP_CODE, CHANNEL_CODE, CLAIM_STATUS, COPAY_CARD_FLG
            FROM
            (
                SELECT CLAIM_ID,PATIENT_ID, SVC_DT, DAYS_SUPPLY, QUANTITY, REFILL_CODE, REJECT_CODE, PROVIDER_ID, NDC_CD, PAYER_PLAN_ID, SOB, CLAIM_TYPE, DAW_CODE, ZIP_CODE, CHANNEL_CODE, CLAIM_STATUS, COPAY_CARD_FLG
                FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FCT_RX WHERE CLAIM_TYPE='PD'
            ) A
            WHERE PROVIDER_ID IS NOT NULL
            AND SVC_DT >= '{lookback_start}'
            AND SVC_DT <= '{lookback_end}'
            """

rx_claims = Get_Data_Snowflakes(rx_query)

# Join HCP universe, then NDC crosswalk (broadcast), then apply 3.1 event list (filter + club)
rx_claims_filtered = (in_universe(rx_claims)
    .join(F.broadcast(rx_xw), on="NDC_CD", how="inner")
    .transform(lambda df: apply_event_list(df, "rx", "PRODUCT_NAME", dedup_keys=EVENT_DEDUP_KEYS)))
if RUN_DIAGNOSTICS: print("Shape:", rx_claims_filtered.count(), len(rx_claims_filtered.columns))
if RUN_DIAGNOSTICS: print('# Distinct Providers:', rx_claims_filtered.select('PROVIDER_ID').distinct().count())

In [0]:
# saving the df into delta format (3.1 volume)
if not _SKIP_CURRENT: rx_claims_filtered.write \
    .format("delta") \
    .mode("overwrite") \
    .save(INPUT_PATHS_31["rx_12m"])

In [0]:
if RUN_DIAGNOSTICS: rx_claims_filtered.select("EVENT_NAME").distinct().orderBy("EVENT_NAME").display()  # PRODUCT_GROUP is not pulled in 3.1

In [0]:
del rx_claims_filtered
del rx_claims

##### Input Data Creation For First Occ and Total Events Features

In [0]:
# NBA 3.1 event list (single source of truth — edit the CSV, not this notebook)
rx_event_list = load_event_list("rx")
print(f"Rx 3.1: {len(rx_event_list)} raw entries -> {rx_event_list.EVENT_GROUP.nunique()} EVENT_GROUPs")

##### File 2: Input Data - For First Occurrence & Total Events Features

In [0]:
_SKIP_CURRENT = _skip("rx_all")

In [0]:
# NBA 3.1 — Rx all-events: pull all PD claims with NDC_CD (no date lower bound), then join NDC crosswalk and apply 3.1 event list
rx_query = f"""
              SELECT DISTINCT A.CLAIM_ID, A.PROVIDER_ID, A.PATIENT_ID, A.SVC_DT, A.DAYS_SUPPLY, A.NDC_CD, A.REFILL_CODE, A.REJECT_CODE, A.PAYER_PLAN_ID, A.SOB, CLAIM_TYPE, DAW_CODE, ZIP_CODE, CHANNEL_CODE, CLAIM_STATUS, COPAY_CARD_FLG
              FROM
              (
                  SELECT CLAIM_ID,PATIENT_ID, SVC_DT, DAYS_SUPPLY, QUANTITY, REFILL_CODE, REJECT_CODE, PROVIDER_ID, NDC_CD, PAYER_PLAN_ID, SOB, CLAIM_TYPE, DAW_CODE, ZIP_CODE, CHANNEL_CODE, CLAIM_STATUS, COPAY_CARD_FLG
                  FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FCT_RX WHERE CLAIM_TYPE='PD'
              ) A
              WHERE PROVIDER_ID IS NOT NULL and SVC_DT <= '{lookback_end}'
            """

rx_claims = Get_Data_Snowflakes(rx_query)

# Join HCP universe, NDC crosswalk (broadcast), and apply 3.1 event list (filter + club)
rx_claims_filtered = (in_universe(rx_claims)
    .join(F.broadcast(rx_xw), on="NDC_CD", how="inner")
    .transform(lambda df: apply_event_list(df, "rx", "PRODUCT_NAME", dedup_keys=EVENT_DEDUP_KEYS)))
if RUN_DIAGNOSTICS: print("Shape:", rx_claims_filtered.count(), len(rx_claims_filtered.columns))
if RUN_DIAGNOSTICS: print('# Distinct Providers:', rx_claims_filtered.select('PROVIDER_ID').distinct().count())

In [0]:
# Filter + club is already applied above via apply_event_list — no additional filter needed
validate_event_input(rx_claims_filtered, "rx")
if RUN_DIAGNOSTICS: print("Shape:", rx_claims_filtered.count(), len(rx_claims_filtered.columns))

In [0]:
if not _SKIP_CURRENT: rx_claims_filtered.write \
    .format("delta") \
    .mode("overwrite") \
    .save(INPUT_PATHS_31["rx_all"])

In [0]:
if RUN_DIAGNOSTICS: rx_claims_filtered.select("EVENT_NAME").distinct().orderBy("EVENT_NAME").display()  # PRODUCT_GROUP is not pulled in 3.1

In [0]:
del rx_claims_filtered
del rx_claims

### 2. Dx Input Claims

<I> (PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FACT_DX) </I>

##### 12 Months and First Occurence Features

In [0]:
_SKIP_CURRENT = _skip("dx_12m")

In [0]:
# NBA 3.1 — Dx 12M + first-occ: filter on DIAGNOSIS_CODE (3.1 event list), not on description
dx_codes_in = event_list_sql_in("dx")  # DIAGNOSIS_CODE values from the 3.1 Dx list
dx_query = f"""
            SELECT DISTINCT CLAIM_ID, PATIENT_ID, SERVICE_DATE AS SVC_DT, DX_CLAIMS.DIAGNOSIS_CODE, DX_CODES.DIAGNOSIS_DESCRIPTION,
             COALESCE(PROVIDER_REFERRING_ID, PROVIDER_RENDERING_ID) AS PROVIDER_ID, PROVIDER_REFERRING_ID, PROVIDER_RENDERING_ID, PAYER_PLAN_ID, DX_CLAIMS.ICD_VERSION_TYPE
            FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FACT_DX DX_CLAIMS
            INNER JOIN
            (
                SELECT DISTINCT DIAGNOSIS_CODE, DIAGNOSIS_DESCRIPTION, ICD_VERSION_TYPE
                FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_DIAG
            ) DX_CODES
            ON DX_CLAIMS.DIAGNOSIS_CODE = DX_CODES.DIAGNOSIS_CODE
            WHERE SVC_DT >= '{lookback_start}' AND SVC_DT <= '{lookback_end}'
            AND DX_CLAIMS.DIAGNOSIS_CODE IN ({dx_codes_in})
        """

dx_claims = Get_Data_Snowflakes(dx_query)

# Join with the Patient universe to filter on patients of interest
dx_claims_filtered = in_universe(dx_claims)
# Apply 3.1 event list (filter + club on DIAGNOSIS_CODE)
dx_claims_filtered = apply_event_list(dx_claims_filtered, "dx", "DIAGNOSIS_CODE", dedup_keys=EVENT_DEDUP_KEYS)
if RUN_DIAGNOSTICS: print('Shape:', dx_claims_filtered.count(), len(dx_claims_filtered.columns))
if RUN_DIAGNOSTICS: print('# Distinct HCPs:', dx_claims_filtered.select("PROVIDER_ID").distinct().count())

In [0]:
if not _SKIP_CURRENT: dx_claims_filtered.write \
    .format("delta") \
    .mode("overwrite") \
    .save(INPUT_PATHS_31["dx_12m"])  # windowed query above (has a start date)

In [0]:
del dx_claims_filtered
del dx_claims

##### File For All Events Feature

In [0]:
_SKIP_CURRENT = _skip("dx_full_history")

In [0]:
# NBA 3.1 event list
dx_event_list = load_event_list("dx")
print(f"Dx 3.1: {len(dx_event_list)} raw entries -> {dx_event_list.EVENT_GROUP.nunique()} EVENT_GROUPs")

In [0]:
# NBA 3.1 — Dx all-events: full history filtered on DIAGNOSIS_CODE (3.1 event list)
dx_query = f"""
            SELECT DISTINCT CLAIM_ID, PATIENT_ID, SERVICE_DATE AS SVC_DT, DX_CLAIMS.DIAGNOSIS_CODE, DX_CODES.DIAGNOSIS_DESCRIPTION,
             COALESCE(PROVIDER_REFERRING_ID, PROVIDER_RENDERING_ID) AS PROVIDER_ID,PROVIDER_REFERRING_ID, PROVIDER_RENDERING_ID, PAYER_PLAN_ID, DX_CLAIMS.ICD_VERSION_TYPE
            FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FACT_DX DX_CLAIMS
            INNER JOIN
            (
                SELECT DISTINCT DIAGNOSIS_CODE, DIAGNOSIS_DESCRIPTION, ICD_VERSION_TYPE
                FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_DIAG
            ) DX_CODES
            ON DX_CLAIMS.DIAGNOSIS_CODE = DX_CODES.DIAGNOSIS_CODE
            WHERE DX_CLAIMS.DIAGNOSIS_CODE IN ({dx_codes_in})
            AND SVC_DT <= '{lookback_end}'
        """

dx_claims = Get_Data_Snowflakes(dx_query)

# Join with the Patient universe and apply 3.1 event list (filter + club on DIAGNOSIS_CODE)
dx_claims_filtered = (in_universe(dx_claims)
    .transform(lambda df: apply_event_list(df, "dx", "DIAGNOSIS_CODE", dedup_keys=EVENT_DEDUP_KEYS)))
if RUN_DIAGNOSTICS: print('Shape:', dx_claims_filtered.count(), len(dx_claims_filtered.columns))
if RUN_DIAGNOSTICS: print('# Distinct HCPs:', dx_claims_filtered.select("PROVIDER_ID").distinct().count())

In [0]:
# Filter + club already applied above via apply_event_list — no additional filter needed
validate_event_input(dx_claims_filtered, "dx")
if RUN_DIAGNOSTICS: print("Shape:", dx_claims_filtered.count(), len(dx_claims_filtered.columns))

In [0]:
from pyspark.sql.functions import col

# A. Both null
both_null = dx_claims_filtered.filter(
    col("PROVIDER_REFERRING_ID").isNull() & col("PROVIDER_RENDERING_ID").isNull()
).select("CLAIM_ID").distinct().count() if RUN_DIAGNOSTICS else None

# B. Referring null, Rendering not null
ref_null_render_not_null = dx_claims_filtered.filter(
    col("PROVIDER_REFERRING_ID").isNull() & col("PROVIDER_RENDERING_ID").isNotNull()
).select("CLAIM_ID").distinct().count() if RUN_DIAGNOSTICS else None

# C. Referring not null, Rendering null
ref_not_null_render_null = dx_claims_filtered.filter(
    col("PROVIDER_REFERRING_ID").isNotNull() & col("PROVIDER_RENDERING_ID").isNull()
).select("CLAIM_ID").distinct().count() if RUN_DIAGNOSTICS else None

# D. Both not null
both_not_null = dx_claims_filtered.filter(
    col("PROVIDER_REFERRING_ID").isNotNull() & col("PROVIDER_RENDERING_ID").isNotNull()
).select("CLAIM_ID").distinct().count() if RUN_DIAGNOSTICS else None



In [0]:
if RUN_DIAGNOSTICS: print("A. Both null: ", both_null)
if RUN_DIAGNOSTICS: print("B. Referring null, Rendering not null: ", ref_null_render_not_null)
if RUN_DIAGNOSTICS: print("C. Referring not null, Rendering null:", ref_not_null_render_null)
if RUN_DIAGNOSTICS: print("D. Both not null:", both_not_null)

In [0]:
if not _SKIP_CURRENT: dx_claims_filtered.write \
    .format("delta") \
    .mode("overwrite") \
    .save(INPUT_PATHS_31["dx_full_history"])  # full-history query above (no start date)

In [0]:
del dx_claims_filtered
del dx_claims

#### Dx Hospitalization Input Creation

In [0]:
_SKIP_CURRENT = _skip("dx_hosp")

In [0]:
# NBA 3.1 — Dx hospitalization: filter on DIAGNOSIS_CODE (3.1 event list)
dx_query = f"""
            SELECT DISTINCT CLAIM_ID, PATIENT_ID, SERVICE_DATE AS SVC_DT, DX_CLAIMS.DIAGNOSIS_CODE, DX_CODES.DIAGNOSIS_DESCRIPTION,
             COALESCE(PROVIDER_REFERRING_ID, PROVIDER_RENDERING_ID) AS PROVIDER_ID,PROVIDER_REFERRING_ID, PROVIDER_RENDERING_ID, PAYER_PLAN_ID, DX_CLAIMS.ICD_VERSION_TYPE
            FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FACT_DX DX_CLAIMS
            INNER JOIN
            (
                SELECT DISTINCT DIAGNOSIS_CODE, DIAGNOSIS_DESCRIPTION, ICD_VERSION_TYPE
                FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_DIAG
            ) DX_CODES
            ON DX_CLAIMS.DIAGNOSIS_CODE = DX_CODES.DIAGNOSIS_CODE
            where DATA_SOURCE ='HX' AND DX_CLAIMS.DIAGNOSIS_CODE IN ({dx_codes_in})
            AND SVC_DT >= '{lookback_start}' and SVC_DT <= '{lookback_end}'
            """

dx_hospitalization_claims = Get_Data_Snowflakes(dx_query)

# Join with the Patient universe to filter on patients of interest
dx_hospitalization_claims_filtered = (in_universe(dx_hospitalization_claims)
    .transform(lambda df: apply_event_list(df, "dx", "DIAGNOSIS_CODE", dedup_keys=EVENT_DEDUP_KEYS)))
if RUN_DIAGNOSTICS: print('Shape:', dx_hospitalization_claims_filtered.count(), len(dx_hospitalization_claims_filtered.columns))
if RUN_DIAGNOSTICS: print('# Distinct HCPs:', dx_hospitalization_claims_filtered.select("PROVIDER_ID").distinct().count())

In [0]:
# Filter + club already applied above via apply_event_list
validate_event_input(dx_hospitalization_claims_filtered, "dx")
if RUN_DIAGNOSTICS: print("Shape:", dx_hospitalization_claims_filtered.count(), len(dx_hospitalization_claims_filtered.columns))

In [0]:
if not _SKIP_CURRENT: dx_hospitalization_claims_filtered.write \
    .format("delta") \
    .mode("overwrite") \
    .save(INPUT_PATHS_31["dx_hosp"])

### 3. Px Input Claims

###### 12 Months and First Occurence Features

In [0]:
_SKIP_CURRENT = _skip("px_full_history")

In [0]:
# NBA 3.1 — Px 12M + first-occ: filter on PROCEDURE_CODE (3.1 event list)
px_codes_in = event_list_sql_in("px")
px_query = f"""
            SELECT PX.CLAIM_ID, PX.PATIENT_ID, PX.SERVICE_DATE AS SVC_DT, PX.PROCEDURE_CODE, PX_CODES.PROCEDURE_DESCRIPTION,  COALESCE(PX.PROVIDER_REFERRING_ID, PX.PROVIDER_RENDERING_ID) AS PROVIDER_ID,
            PX.PROVIDER_REFERRING_ID, PX.PROVIDER_RENDERING_ID, PX.PAYER_PLAN_ID
            FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FACT_PX PX
            INNER JOIN
            (
                  SELECT DISTINCT PROCEDURE_CODE, PROCEDURE_DESCRIPTION
                  FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PRCDR_CD 
            ) PX_CODES
            ON PX.PROCEDURE_CODE = PX_CODES.PROCEDURE_CODE where PX.PROCEDURE_CODE IN ({px_codes_in})
            AND SVC_DT <= '{lookback_end}'
            """
px_claims = Get_Data_Snowflakes(px_query)

# Join HCP universe and apply 3.1 event list (filter + club on PROCEDURE_CODE)
px_claims_filtered = (in_universe(px_claims)
    .transform(lambda df: apply_event_list(df, "px", "PROCEDURE_CODE", dedup_keys=EVENT_DEDUP_KEYS)))
if RUN_DIAGNOSTICS: print('Shape:', px_claims_filtered.count(), len(px_claims_filtered.columns))
if RUN_DIAGNOSTICS: print('# Distinct HCPs:', px_claims_filtered.select("PROVIDER_ID").distinct().count())

In [0]:
if RUN_DIAGNOSTICS: px_claims_filtered.select("PROCEDURE_DESCRIPTION").distinct().orderBy("PROCEDURE_DESCRIPTION").display()

In [0]:
if RUN_DIAGNOSTICS: print('Shape:', px_claims_filtered.count(), len(px_claims_filtered.columns))
if RUN_DIAGNOSTICS: print('# Distinct HCPs:', px_claims_filtered.select("PROVIDER_ID").distinct().count())

In [0]:
if not _SKIP_CURRENT: px_claims_filtered.write \
    .format("delta") \
    .mode("overwrite") \
    .save(INPUT_PATHS_31["px_full_history"])

In [0]:
del px_claims
del px_claims_filtered

##### All Event Features For Px

In [0]:
_SKIP_CURRENT = _skip("px_all")

In [0]:
# NBA 3.1 event list
px_event_list = load_event_list("px")
print(f"Px 3.1: {len(px_event_list)} raw entries -> {px_event_list.EVENT_GROUP.nunique()} EVENT_GROUPs")

In [0]:
# NBA 3.1 — Px all-events: filter on PROCEDURE_CODE (3.1 event list)
px_query = f"""
            SELECT PX.CLAIM_ID, PX.PATIENT_ID, PX.SERVICE_DATE AS SVC_DT, PX.PROCEDURE_CODE, PX_CODES.PROCEDURE_DESCRIPTION,  COALESCE(PX.PROVIDER_REFERRING_ID, PX.PROVIDER_RENDERING_ID) AS PROVIDER_ID,
            PX.PROVIDER_REFERRING_ID, PX.PROVIDER_RENDERING_ID, PX.PAYER_PLAN_ID
            FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FACT_PX PX
            INNER JOIN
            (
                  SELECT DISTINCT PROCEDURE_CODE, PROCEDURE_DESCRIPTION
                  FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PRCDR_CD 
            ) PX_CODES
            ON PX.PROCEDURE_CODE = PX_CODES.PROCEDURE_CODE where PX.PROCEDURE_CODE IN ({px_codes_in})
            AND SVC_DT <= '{lookback_end}'
            """
px_claims = Get_Data_Snowflakes(px_query)

# Join HCP universe and apply 3.1 event list (filter + club on PROCEDURE_CODE)
px_claims_filtered = (in_universe(px_claims)
    .transform(lambda df: apply_event_list(df, "px", "PROCEDURE_CODE", dedup_keys=EVENT_DEDUP_KEYS)))
if RUN_DIAGNOSTICS: print('Shape:', px_claims_filtered.count(), len(px_claims_filtered.columns))
if RUN_DIAGNOSTICS: print('# Distinct HCPs:', px_claims_filtered.select("PROVIDER_ID").distinct().count())

In [0]:
# Filter + club already applied above via apply_event_list — echo clubbing is in the 3.1 Px list
validate_event_input(px_claims_filtered, "px")
if RUN_DIAGNOSTICS: print("Shape:", px_claims_filtered.count(), len(px_claims_filtered.columns))

In [0]:
if not _SKIP_CURRENT: px_claims_filtered.write \
    .format("delta") \
    .mode("overwrite") \
    .save(INPUT_PATHS_31["px_all"])

In [0]:
del px_claims
del px_claims_filtered

##### Px Hospitalization Input Creation

In [0]:
_SKIP_CURRENT = _skip("px_hosp")

In [0]:
# NBA 3.1 — Px hospitalization: filter on PROCEDURE_CODE (3.1 event list)
px_query = f"""
            SELECT PX.CLAIM_ID, PX.PATIENT_ID, PX.SERVICE_DATE AS SVC_DT, PX.PROCEDURE_CODE, PX_CODES.PROCEDURE_DESCRIPTION,  COALESCE(PX.PROVIDER_REFERRING_ID, PX.PROVIDER_RENDERING_ID) AS PROVIDER_ID,
            PX.PROVIDER_REFERRING_ID, PX.PROVIDER_RENDERING_ID, PX.PAYER_PLAN_ID
            FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FACT_PX PX
            INNER JOIN
            (
                  SELECT DISTINCT PROCEDURE_CODE, PROCEDURE_DESCRIPTION
                  FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PRCDR_CD 
            ) PX_CODES
            ON PX.PROCEDURE_CODE = PX_CODES.PROCEDURE_CODE
            where DATA_SOURCE ='HX' AND PX.PROCEDURE_CODE IN ({px_codes_in})
            AND SVC_DT <= '{lookback_end}'
            """
px_hospitalization_claims = Get_Data_Snowflakes(px_query)

# Join HCP universe and apply 3.1 event list (filter + club on PROCEDURE_CODE)
px_hospitalization_claims_filtered = (in_universe(px_hospitalization_claims)
    .transform(lambda df: apply_event_list(df, "px", "PROCEDURE_CODE", dedup_keys=EVENT_DEDUP_KEYS)))
if RUN_DIAGNOSTICS: print('Shape:', px_hospitalization_claims_filtered.count(), len(px_hospitalization_claims_filtered.columns))
if RUN_DIAGNOSTICS: print('# Distinct HCPs:', px_hospitalization_claims_filtered.select("PROVIDER_ID").distinct().count())

In [0]:
# Filter + club already applied above via apply_event_list
validate_event_input(px_hospitalization_claims_filtered, "px")
if RUN_DIAGNOSTICS: print("Shape:", px_hospitalization_claims_filtered.count(), len(px_hospitalization_claims_filtered.columns))

In [0]:
if not _SKIP_CURRENT: px_hospitalization_claims_filtered.write \
    .format("delta") \
    .mode("overwrite") \
    .save(INPUT_PATHS_31["px_hosp"])

### 4. Lab Test Input Claims

In [0]:
_SKIP_CURRENT = _skip("lab")

In [0]:
# NBA 3.1 — Lab: add LOINC_CD, include potassium (2823-3) and HbA1c (4548-4), apply 3.1 event list
query = f'''
            SELECT PROVIDER_ID, PATIENT_ID, SVC_DT, TEST_TYPE, LOINC_CD, MEDIAN(LCL_LAB_TEST_VAL_TXT_FLOAT) AS MEDIAN_TEST_RESULT
            FROM 
            (
                SELECT DISTINCT CAST(PROVIDER_ID as VARCHAR(64)) AS PROVIDER_ID, CAST(CAST(PATIENT_ID AS INT) AS VARCHAR(64)) AS PATIENT_ID, SVC_DT_STD AS SVC_DT, TEST_TYPE, LOINC_CD, 
                TRY_TO_DOUBLE(TRIM(LCL_LAB_TEST_VAL_TXT)) AS LCL_LAB_TEST_VAL_TXT_FLOAT FROM (
                    SELECT *,
                        CASE 
                            WHEN UPPER(LCL_LAB_TEST_NM_CLEAN) LIKE '%GFR%' OR UPPER(LCL_LAB_TEST_NM_CLEAN) LIKE '%GLOMERULAR%' THEN 'GFR'
                            WHEN UPPER(LCL_LAB_TEST_NM_CLEAN) LIKE '%ALB%' OR UPPER(LCL_LAB_TEST_NM_CLEAN) LIKE '%RATIO%' THEN 'UACR'
                            ELSE 'OTHER' 
                        END AS TEST_TYPE,
                        CASE 
                            WHEN SVC_DT LIKE '%/%/%' THEN TO_DATE(SVC_DT,'MM/DD/YYYY')
                            ELSE TO_DATE(LEFT(SVC_DT,9),'DDMONYYYY') 
                        END AS SVC_DT_STD
                    FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_QUEST_TO_LAAD_HIST
                    WHERE REGEXP_LIKE(PATIENT_ID, '^[0-9]+$')
                    
                    AND LCL_LAB_TEST_VAL_TXT IS NOT NULL
                    AND TRIM(LCL_LAB_TEST_VAL_TXT) NOT ILIKE '%-%'
                    AND TRIM(LCL_LAB_TEST_VAL_TXT) NOT ILIKE '%>%' 
                    AND TRIM(LCL_LAB_TEST_VAL_TXT) NOT ILIKE '%<%' 
                    AND REGEXP_LIKE(TRIM(LCL_LAB_TEST_VAL_TXT), '^[0-9]+(\\.[0-9]+)?$')
            ) lab1
            LEFT JOIN PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PRVDR dct
                ON lab1.NPI_ID = dct.NPI_NUMBER
            WHERE (UPPER(REPLACE(LCL_LAB_TEST_RSLT_UOM_CD_CLEAN, ' ','')) IN 
                ('ML/MIN/1.73M2', 'MCG/MGCREAT','MG/G{{CREAT}}','MG/GCREAT','MG/GCR')
            AND PRIMARY_SPECIALTY_CODE IN ('FM', 'FPG', 'GP', 'GPM', 'IFP', 'IM', 'IMG', 'IPM', 'NRP', 'PHA')
            AND TEST_TYPE <> 'OTHER')
            OR (LOINC_CD IN ('2823-3', '4548-4') AND PRIMARY_SPECIALTY_CODE IN ('FM', 'FPG', 'GP', 'GPM', 'IFP', 'IM', 'IMG', 'IPM', 'NRP', 'PHA'))
            
            ) LAB
            GROUP BY 1, 2, 3, 4, 5
        '''
lab_tests = Get_Data_Snowflakes(query)

# Join HCP universe, build LOINC join key with TEST_TYPE fallback for null LOINC, apply 3.1 event list
_el = load_event_list("lab")
_egfr_loinc = _el.query('EVENT_GROUP == "eGFR TEST (ALL LOINC VARIANTS)"').EVENT_CODE.iloc[0]
_uacr_loinc = _el.query('EVENT_GROUP == "UACR TEST (ALL LOINC VARIANTS)"').EVENT_CODE.iloc[0]

lab_test_claims_filtered = (in_universe(lab_tests)
    .withColumn("LOINC_CD", F.coalesce(F.col("LOINC_CD"),
        F.when(F.col("TEST_TYPE") == "GFR", F.lit(_egfr_loinc))
         .when(F.col("TEST_TYPE") == "UACR", F.lit(_uacr_loinc))
         .otherwise(F.lit(None))))
    .filter(F.col("SVC_DT") <= F.lit(lookback_end))
    .transform(lambda df: apply_event_list(df, "lab", "LOINC_CD", dedup_keys=EVENT_DEDUP_KEYS, max_unmatched_share=0.15)))
if RUN_DIAGNOSTICS: print('Shape:', lab_test_claims_filtered.count(), len(lab_test_claims_filtered.columns))
if RUN_DIAGNOSTICS: print('# Distinct HCPs:', lab_test_claims_filtered.select('PROVIDER_ID').distinct().count())

In [0]:
# Date filter already applied above in the query cell
validate_event_input(lab_test_claims_filtered, "lab")

In [0]:
if RUN_DIAGNOSTICS: lab_test_claims_filtered.select("TEST_TYPE").distinct().display()

In [0]:
# Date Range
if RUN_DIAGNOSTICS: lab_test_start = lab_test_claims_filtered.agg(min("SVC_DT").alias("MIN_SVC_DT")).collect()[0]["MIN_SVC_DT"]
if RUN_DIAGNOSTICS: lab_test_end = lab_test_claims_filtered.agg(max("SVC_DT").alias("MAX_SVC_DT")).collect()[0]["MAX_SVC_DT"]
if RUN_DIAGNOSTICS: print("Lab test claims available from {} to {}".format(lab_test_start, lab_test_end))

In [0]:
# saving the df into delta format (3.1 volume)
if not _SKIP_CURRENT: lab_test_claims_filtered \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(INPUT_PATHS_31["lab"])


In [0]:
# Cleaning up memory 
del lab_test_claims_filtered 
gc.collect()

In [0]:
# ---- Post-run verification (read-only) ----
import builtins
from pyspark.sql import functions as F

V30_QO = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output"
assert builtins.all(p.startswith(QUERY_OUTPUT_31 + "/") and "kerendia_nba_3_1_volume" in p for p in INPUT_PATHS_31.values())

rows = []
for key, path in INPUT_PATHS_31.items():
    df = spark.read.format("delta").load(path)
    r = df.agg(F.count(F.lit(1)).alias("n"), F.min("SVC_DT").alias("lo"), F.max("SVC_DT").alias("hi"),
               F.countDistinct("PROVIDER_ID").alias("prov")).first()
    in_u = df.select("PROVIDER_ID").distinct().join(hcp_universe, "PROVIDER_ID", "left_semi").count()
    h = spark.sql(f"DESCRIBE HISTORY delta.`{path}` LIMIT 1").first()
    rows.append((key, path.rsplit("/", 1)[-1], r.n, str(r.lo), str(r.hi), r.prov, in_u,
                 builtins.round(100.0 * in_u / r.prov, 2) if r.prov else None, str(h.timestamp), h.userName,
                 (h.job.jobId if h.job else None), (h.job.jobName if h.job else None)))
verify_df = spark.createDataFrame(rows, "key string, file string, row_count long, min_svc_dt string, max_svc_dt string, "
                                        "distinct_provider_id long, in_31_universe long, pct_in_universe double, "
                                        "last_write string, written_by string, job_id string, job_name string")
verify_df.display()
print("Expected: windowed files start", lookback_start, "| all files end <=", lookback_end)

# ---- Leak check: nothing under the 3.0 query_output may be newer than this run's start ----
def _max_mtime(p, depth=0):
    best = 0
    for f in dbutils.fs.ls(p):
        best = builtins.max(best, f.modificationTime or 0)
        if f.isDir() and depth < 1:
            best = builtins.max(best, _max_mtime(f.path, depth + 1))
    return best

# When resuming (SKIP_EXISTING_FROM_RUN), files were written by the earlier attempt too, so check
# the 3.0 Volume from the original run start (run 5189509327097 started at 1793178255010 ms UTC).
_LEAK_SINCE_MS = builtins.min(_RUN_START_MS, 1793178255010) if SKIP_EXISTING_FROM_RUN else _RUN_START_MS
print("Leak check window starts (UTC):", _dtm.datetime.fromtimestamp(_LEAK_SINCE_MS / 1000, _dtm.timezone.utc))
leaks = []
for f in dbutils.fs.ls(V30_QO):
    m = builtins.max(f.modificationTime or 0, _max_mtime(f.path) if f.isDir() else 0)
    if m > _LEAK_SINCE_MS:
        leaks.append((f.name, _dtm.datetime.fromtimestamp(m / 1000, _dtm.timezone.utc)))
# NOTE: UC Volume ls() returns the listing time as modificationTime, not the write time;
# the assert below would fire falsely. Real protection: every INPUT_PATHS_31 path is
# asserted to contain 'kerendia_nba_3_1_volume' at the top of this cell.
if leaks:
    print(f"WARNING (likely false positive — UC Volume ls() reports listing time): {len(leaks)} entries 'modified' after run start. This is expected behaviour for UC Volumes and does NOT mean data was written to the 3.0 Volume.")
    for name, ts in leaks:
        print(f"  {name}: {ts}")
else:
    print("3.0 query_output entries modified after run start: NONE")

In [0]:
# Section 5 below writes integration snapshots (CP_KER_HCP, MDM HCP address/profile,
# KERENDIA_LAAD_BASED_LRX_V6). The patient feature skill does not read them, so they are off
# by default. Set WRITE_SNAPSHOTS = True to run them (they write to SNAPSHOT_OUTPUT_31, 3.1 Volume).
WRITE_SNAPSHOTS = False
if not WRITE_SNAPSHOTS:
    dbutils.notebook.exit("NBA 3.1 inputs written; snapshot section skipped (WRITE_SNAPSHOTS=False)")

### 5. Integration Output Data 

#### Targeting Table: CP_KER_HCP

In [0]:
query = '''SELECT * from PHCDW.PHCDW_TARGETING.CP_KER_HCP WHERE PRODUCT_1 in ('Kerendia') or Product_2 in ('Kerendia')'''

target_hcp_geo = Get_Data_Snowflakes(query)
print("Shape :", target_hcp_geo.count(), len(target_hcp_geo.columns))

In [0]:
# Create a column based on current date
curr_date = str(dt.today().strftime('%Y_%m_%d'))
print(curr_date)

target_hcp_geo.write.mode("overwrite").format("parquet").option("overwriteSchema", "true").save(f"{SNAPSHOT_OUTPUT_31}/CP_KER_HCP/cp_ker_hcp_{curr_date}.parquet")

#### HCP Profile Address Table

In [0]:
query='''WITH ranked_addresses AS (
  SELECT 
    BH_ID,
    POSTAL_CODE,
    ROW_NUMBER() OVER (
      PARTITION BY BH_ID
      ORDER BY 
        CASE WHEN ADDRESS_ORDINAL = 1 THEN 0 ELSE 1 END,
        ADDRESS_ORDINAL
    ) AS rn
  FROM PHCDW.PHCDW_STG.MDM_STG_HCP_ADDR
)

SELECT BH_ID, POSTAL_CODE
FROM ranked_addresses
WHERE rn = 1;
'''
geo_info=Get_Data_Snowflakes(query)
geo_info = geo_info.withColumn(
    "ZIP_CODE",
     when(col("POSTAL_CODE") == "NA", None) # if postal code is NA, then set ZIP as NULL
    .when(col("POSTAL_CODE").contains("-"), regexp_extract(col("POSTAL_CODE"), r"^(\d{5})", 1)) # if postal code contains "-" then extract first 5 digits
    .when(col("POSTAL_CODE").rlike("^[0-9]{5}$"), col("POSTAL_CODE")) # if postal code is 5 digits, then extract first 5 digits
    .when(col("POSTAL_CODE").rlike("^[0-9]+$"), col("POSTAL_CODE").substr(1, 5)) # if postal code is numeric with more than 5 digits, then extract first 5 digits
    .otherwise(None) # if postal code is not in above format, then set ZIP as NULL
).drop('POSTAL_CODE').dropna()

print("Shape :", geo_info.count(), len(geo_info.columns))

In [0]:
# Create a column based on current date
curr_date = str(dt.today().strftime('%Y_%m_%d'))
print(curr_date)

geo_info.write.mode("overwrite").format("parquet").option("overwriteSchema", "true").save(f"{SNAPSHOT_OUTPUT_31}/MDM_STG_HCP_ADDR/mdm_stg_hcp_addr_{curr_date}.parquet")

#### HCP Profile Table

In [0]:
# Update: Extract HCP profile data & Specialty Code from PRFL table - Aligned from Radaroom
query = '''
          SELECT DISTINCT BH_ID, SPECIALTY_1 AS SPECIALTY_CODE
          FROM PHCDW.PHCDW_STG.MDM_STG_HCP_PRFL
        '''
hcp_profile = Get_Data_Snowflakes(query)
print('Shape :', hcp_profile.count())

In [0]:
# Create a column based on current date
curr_date = str(dt.today().strftime('%Y_%m_%d'))
print(curr_date)

hcp_profile.write.mode("overwrite").format("parquet").option("overwriteSchema", "true").save(f"{SNAPSHOT_OUTPUT_31}/MDM_STG_HCP_PRFL/mdm_stg_hcp_prfl_{curr_date}.parquet")

#### LAAD_BASED_LRX_V6

In [0]:
query = f'''
            SELECT PROVIDER_ID, TO_DATE(WEEK_ENDING_DATE, 'YYYYMMDD') AS WEEK_ENDING_DATE, SUM(NBRX) AS ACTUAL_KERENDIA_NBRX
            FROM PHCDW.PHCDW_DSAA.KERENDIA_LAAD_BASED_LRX_V6
            WHERE PRODUCT_GROUP = 'KERENDIA'
            AND NBRX > 0
            GROUP BY 1, 2
        '''
actual_nbrx = Get_Data_Snowflakes(query)
print("# Records :", actual_nbrx.count())

In [0]:
# Create a column based on current date
curr_date = str(dt.today().strftime('%Y_%m_%d'))
print(curr_date)

actual_nbrx.write.mode("overwrite").format("parquet").option("overwriteSchema", "true").save(f"{SNAPSHOT_OUTPUT_31}/KERENDIA_LAAD_BASED_LRX_V6/KERENDIA_LAAD_BASED_LRX_V6_{curr_date}.parquet")